# MIT805 Group Project - Group 19
## Part 1 - Exploratory Data Analysis

### Imports and Setup

In [ ]:
# Colab setup

#from google.colab import drive
#drive.mount('/content/drive')

In [4]:
# Imports

import os
import json
import math
import warnings
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from pathlib import Path
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, IntegerType, DoubleType, TimestampType)


In [6]:
# Paths

PROJECT_ROOT = Path.cwd()

RAW_DIR   = PROJECT_ROOT / "data" / "working"
CLEAN_DIR = PROJECT_ROOT / "data" / "cleaned_monthly"
FIG_DIR   = PROJECT_ROOT / "figures"
OUT_DIR   = PROJECT_ROOT / "output"

FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

spark = (SparkSession.builder
         .appName("MIT805 Part1 EDA")
         .config("spark.driver.memory", "6g")
         .config("spark.sql.shuffle.partitions", "24")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

STATS = {}

print("project:", PROJECT_ROOT)
print("Spark  :", spark.version)

project: /Users/shreyabharat/Projects/MIT805-GroupProject
Spark  : 4.2.0


In [ ]:
# Figure style

BLUE = "#0000ff"
GREEN = "#00963f"
RED = "#ff0000"
GREY =  "#52514e"

plt.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.axisbelow": True
})

def thousands(x, _=None):
    if x >= 1e6: return f"{x/1e6:.0f}M"
    if x >= 1e3: return f"{x/1e3:.0f}k"
    return f"{x:.0f}"

def save(fig, name, note):
    fig.text(0.02, -0.04, note, fontsize=7, color="#898781")
    fig.savefig(FIG_DIR / name)
    plt.close(fig)
    print("saved:", name)

### Data loading

In [ ]:
raw_files = sorted(RAW_DIR.glob("yellow_trip_*.parquet"))
month_dirs = sorted(d for d in CLEAN_DIR.glob("yellow_tripdata_*_cleaned")
                    if any(d.glob("*.parquet")))

months = [d.name.replace("yellow_tripdata_","").replace("_cleaned","")
          for d in month_dirs]
paths = [str(d) for d in month_dirs]

print(f"downloaded: {len(raw_files)} months")
print(f"cleaned: {len(months)} months, {months[0]} to {months[-1]}")